# Experiment Notebook for AE and VAE Models

This notebook is for experimenting with the Autoencoder (AE) and Variational Autoencoder (VAE) models. It imports from the modular project files to keep things clean and reusable.

In [ ]:
# Import necessary libraries and project modules
import os
import pathlib
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from models import Autoencoder, VariationalAutoencoder
from utils.dataloader import create_image_dataset, create_image_label_dataset
from utils.viz import plot_reconstructions, plot_training_history, plot_latent_space
from utils.losses import reconstruction_loss, kl_divergence_loss

In [ ]:
# Experiment parameters
DATA_DIR = './medical-mnist'  # Update this path
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32
LATENT_DIM = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
CLASS_NAMES = ['AbdomenCT', 'BreastMRI', 'ChestCT', 'CXR', 'Hand', 'HeadCT']  # For latent space if labeled

In [ ]:
# Load dataset
dataset = create_image_dataset(DATA_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True)
labeled_dataset = create_image_label_dataset(DATA_DIR, CLASS_NAMES, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False)
print(f"Datasets loaded with batch size {BATCH_SIZE}")

In [ ]:
# Build and train Autoencoder
ae = Autoencoder(latent_dim=LATENT_DIM, input_shape=(*IMAGE_SIZE, 1))
ae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')

print("Training Autoencoder...")
ae_history = ae.fit(dataset, epochs=EPOCHS, verbose=1)

# Plot training history
plot_training_history(ae_history)

In [ ]:
# Visualize AE reconstructions
plot_reconstructions(ae, dataset, num_images=8)

In [ ]:
# Build and train Variational Autoencoder
vae = VariationalAutoencoder(latent_dim=LATENT_DIM, input_shape=(*IMAGE_SIZE, 1))
vae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE))

print("Training Variational Autoencoder...")
vae_history = vae.fit(dataset, epochs=EPOCHS, verbose=1)

# Plot training history
plot_training_history(vae_history)

In [ ]:
# Visualize VAE reconstructions
plot_reconstructions(vae, dataset, num_images=8)

In [ ]:
# Visualize VAE latent space (requires labeled dataset)
plot_latent_space(vae, labeled_dataset, CLASS_NAMES, num_points=1000)

In [ ]:
# Generate samples from VAE latent space
import numpy as np
import matplotlib.pyplot as plt

# Sample from standard normal
z = tf.random.normal(shape=(16, LATENT_DIM))
generated = vae.decode(z, apply_sigmoid=True)

# Plot generated samples
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i in range(16):
    ax = axes[i // 4, i % 4]
    ax.imshow(generated[i].numpy().squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Generated Samples from VAE')
plt.show()